![](https://wherobots.com/wp-content/uploads/2023/12/Inline-Blue_Black_onWhite@3x.png)

# DBSCAN 
DBSCAN is a popular algorithm for finding clusters of spatial data. It identifies core points that have enough (defined by the user) neighbors within some distance (also user defined). Points that are not core points but are within the distance of a core point are considered border points of the cluster. Points that are not core points and are not within the distance of a core point are considered outliers and not part of any cluster.

The algorithm requires two parameters:
* epsilon - the maximum distance between two points for them to be considered connected/related
* minPoints - the minimum number of neighbor points (as determined by epsilon) a point must have to be a core point.

In this example, we will generate some random data and use DBSCAN to cluster the data. We will then visualize the clusters using a scatter plot.

This demo is derived from the [scikit-learn DBSCAN demo](https://scikit-learn.org/stable/auto_examples/cluster/plot_dbscan.html).

We begin by installing sklearn to generate gaussian clusters as input data

In [ ]:
!pip install scikit-learn


# Define Sedona Context

In [ ]:
from sedona.spark import SedonaContext

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

# Data Generation
We generate some data using sklearn's make_blobs function. The data consists of 750 points with 3 clusters. We then visualize the data in pyplot

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

centers = [[1, 1], [-1, -1], [1, -1]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=0.4, random_state=0
)

X = StandardScaler().fit_transform(X)

plt.scatter(X[:, 0], X[:, 1])
plt.show()

## Clustering
We use the DBSCAN implementation in Wherobots to cluster the data. We set epsilon to 0.3 and minPoints to 10.

Our DBSCAN does not return outliers by default so we include those in the output by setting `include_outliers=True`.

In [ ]:
import pyspark.sql.functions as f
from sedona.sql.st_constructors import ST_MakePoint
from sedona.stats.clustering.dbscan import dbscan

df = sedona.createDataFrame(X).select(ST_MakePoint("_1", "_2").alias("geometry"))
clusters_df = dbscan(df, 0.3, 10, include_outliers=True)


In [ ]:
clusters_df.show()

## Visualization
We visualize the clusters using geopandas. Some manipulations are made to the data to improve the clarity of the visualization.

In [ ]:
import geopandas as gpd
import pyspark.sql.types as t

pdf = (clusters_df
       .withColumn("isCore", (f.col("isCore").cast(t.IntegerType()) + 1) * 40)
       .withColumn("cluster", f.hash("cluster").cast(t.StringType()))
       .toPandas()
      )
gdf = gpd.GeoDataFrame(pdf, geometry="geometry")

gdf.plot(
    figsize=(10, 8),
    column="cluster",
    markersize=gdf['isCore'],
    edgecolor='lightgray',
)